In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# 1. Dataset Generation
def generate_exponential_dataset(n_samples=5000, seed=42):
    np.random.seed(seed)
    u = np.random.uniform(0, 1, n_samples)
    x = 7.5 * np.tanh(4 * (u - 0.5)) - 2.5
    y = np.exp(x)
    return x.reshape(-1, 1), y.reshape(-1, 1)

# 2. SVDLinear Layer without pruning (full rank at runtime)
class SVDLinear(nn.Module):
    def __init__(self, in_features, out_features, rank):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank

        # Initialize full weight then decompose
        W = torch.empty(out_features, in_features)
        nn.init.kaiming_uniform_(W, a=np.sqrt(5))
        U_full, S_full, V_full = torch.svd(W)
        # Keep initial full-rank factors
        self.U = nn.Parameter(U_full[:, :rank].clone())   # (out_features, rank)
        self.S = nn.Parameter(S_full[:rank].clone())      # (rank,)
        self.V = nn.Parameter(V_full[:, :rank].clone())   # (in_features, rank)
        self.bias = nn.Parameter(torch.zeros(out_features))
        # Optional BatchNorm to stabilize training
        self.bn = nn.BatchNorm1d(out_features)

    def forward(self, x):
        # Always use full rank factors (no pruning)
        V_act = self.V      # (in_features, rank)
        S_act = self.S      # (rank,)
        U_act = self.U      # (out_features, rank)

        # Forward pass: x @ V * S @ U^T
        Vx = x @ V_act       # (batch, rank)
        SVx = Vx * S_act     # (batch, rank)
        out = SVx @ U_act.t() # (batch, out_features)
        out = out + self.bias
        # BatchNorm
        out = self.bn(out)
        return out

# 3. Network Definitions
class SVDNet(nn.Module):
    def __init__(self, input_dim=1, hidden_dims=[256,256,128,128,128,64,32], rank=8):
        super().__init__()
        dims = [input_dim] + hidden_dims + [1]
        self.layers = nn.ModuleList()
        for i in range(len(dims)-1):
            r = min(dims[i], dims[i+1],rank)
            self.layers.append(SVDLinear(dims[i], dims[i+1], r))

    def forward(self, x):
        for layer in self.layers[:-1]:
            x = torch.relu(layer(x))
        return self.layers[-1](x)

class BaselineNet(nn.Module):
    def __init__(self, input_dim=1, hidden_dims=[256,256,128,128,128,64,32]):
        super().__init__()
        layers = []
        dims = [input_dim] + hidden_dims
        for i in range(len(dims)-1):
            layers.append(nn.Linear(dims[i], dims[i+1]))
            layers.append(nn.BatchNorm1d(dims[i+1]))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(dims[-1], 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# 4. Training Loop
def train_models(X_train, y_train, X_test, y_test, epochs=200, log_interval=20):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    X_train = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_train = torch.tensor(y_train, dtype=torch.float32, device=device)
    X_test = torch.tensor(X_test, dtype=torch.float32, device=device)
    y_test = torch.tensor(y_test, dtype=torch.float32, device=device)

    baseline = BaselineNet().to(device)
    svdnet = SVDNet(rank=32).to(device)

    optim_b = optim.AdamW(baseline.parameters(), lr=1e-3, weight_decay=1e-4)
    optim_s = optim.AdamW(svdnet.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.MSELoss()

    history = {'epoch': [],
               'train_loss_b': [], 'test_loss_b': [],
               'train_loss_s': [], 'test_loss_s': []}

    for epoch in range(1, epochs+1):
        # Baseline training step
        baseline.train()
        optim_b.zero_grad()
        pred_b = baseline(X_train)
        loss_b = criterion(pred_b, y_train)
        loss_b.backward()
        optim_b.step()

        # SVD training step
        svdnet.train()
        optim_s.zero_grad()
        pred_s = svdnet(X_train)
        loss_s_main = criterion(pred_s, y_train)
       
        loss_s = loss_s_main 
        loss_s.backward()
        optim_s.step()

        # Logging
        if epoch % log_interval == 0:
            baseline.eval()
            svdnet.eval()
            with torch.no_grad():
                tb = criterion(baseline(X_train), y_train).item()
                vb = criterion(baseline(X_test), y_test).item()
                ts = criterion(svdnet(X_train), y_train).item()
                vs = criterion(svdnet(X_test), y_test).item()

            print(f"Epoch {epoch}/{epochs}"
                  f" | Base Train: {tb:.6f}, Base Test: {vb:.6f}"
                  f" | SVD Train: {ts:.6f}, SVD Test: {vs:.6f}")

            history['epoch'].append(epoch)
            history['train_loss_b'].append(tb)
            history['test_loss_b'].append(vb)
            history['train_loss_s'].append(ts)
            history['test_loss_s'].append(vs)

    return history

# 5. Main Execution
if __name__ == "__main__":
    # Generate and split data
    X, y = generate_exponential_dataset()
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    Xs = scaler_X.fit_transform(X)
    ys = scaler_y.fit_transform(y)

    X_train, X_test, y_train, y_test = train_test_split(
        Xs, ys, test_size=0.2, random_state=42
    )

    # Train and log
    history = train_models(X_train, y_train, X_test, y_test)

    # Save history
    import pandas as pd
    df = pd.DataFrame(history)
    df.to_csv('training_history.csv', index=False)
    print("Training complete, history saved to training_history.csv")


Epoch 20/200 | Base Train: 0.801455, Base Test: 0.809828 | SVD Train: 0.925080, SVD Test: 0.933222
Epoch 40/200 | Base Train: 0.059589, Base Test: 0.059929 | SVD Train: 0.646220, SVD Test: 0.644891
Epoch 60/200 | Base Train: 0.003609, Base Test: 0.003570 | SVD Train: 0.203043, SVD Test: 0.203558
Epoch 80/200 | Base Train: 0.001316, Base Test: 0.001186 | SVD Train: 0.009777, SVD Test: 0.009696
Epoch 100/200 | Base Train: 0.000175, Base Test: 0.000168 | SVD Train: 0.000425, SVD Test: 0.000429
Epoch 120/200 | Base Train: 0.000038, Base Test: 0.000038 | SVD Train: 0.000157, SVD Test: 0.000158
Epoch 140/200 | Base Train: 0.000018, Base Test: 0.000018 | SVD Train: 0.000130, SVD Test: 0.000131
Epoch 160/200 | Base Train: 0.000016, Base Test: 0.000016 | SVD Train: 0.000086, SVD Test: 0.000087
Epoch 180/200 | Base Train: 0.000012, Base Test: 0.000012 | SVD Train: 0.000073, SVD Test: 0.000074
Epoch 200/200 | Base Train: 0.000008, Base Test: 0.000009 | SVD Train: 0.000061, SVD Test: 0.000062
Trai